# B2B MRO Supply Chain Analytics
## End-to-End Procurement & Customer Engagement

**Author:** Marziyeh Eslamparasti — Business Analyst | Hamburg, Germany  
**Tools:** Python · Pandas · DuckDB · scikit-learn  
**Data:** Synthetic dataset modelling a B2B industrial tools & MRO distributor — parameters derived from real procurement experience.

---
**Two core business problems this analysis addresses:**
1. **Basket size** — which customer segments buy more per order, and what drives it?
2. **End-to-end order management** — where do delays happen, and who is most affected?


## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import duckdb
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

# Load all tables — named descriptively so it is clear what each contains
customer_orders  = pd.read_csv('customer_orders.csv',  parse_dates=['order_date'])
customers        = pd.read_csv('customers.csv')
mro_products     = pd.read_csv('products.csv')
supplier_master  = pd.read_csv('suppliers.csv')
supplier_orders  = pd.read_csv('supplier_orders.csv',  parse_dates=['order_date'])

print(f"Customer orders : {len(customer_orders):,} rows")
print(f"Customers       : {len(customers):,}")
print(f"Products (SKUs) : {len(mro_products):,}")
print(f"Suppliers       : {len(supplier_master):,}")
print(f"Supplier orders : {len(supplier_orders):,} rows")


## 2. Key Performance Indicators

In [ ]:
# Core KPIs — calculated once and referenced throughout the notebook
total_orders       = len(customer_orders)
total_revenue      = customer_orders['order_value'].sum()
ontime_rate        = customer_orders['on_time'].mean() * 100
late_orders        = (customer_orders['on_time'] == 0).sum()
avg_basket_size    = customer_orders['basket_size'].mean()
active_customers   = customer_orders['customer_id'].nunique()

print("── Key Performance Indicators ──────────────────")
print(f"  Total orders     : {total_orders:,}")
print(f"  Total revenue    : €{total_revenue:,.0f}")
print(f"  On-time rate     : {ontime_rate:.1f}%")
print(f"  Late orders      : {late_orders:,}")
print(f"  Avg basket size  : {avg_basket_size:.1f} items/order")
print(f"  Active customers : {active_customers:,}")


## 3. Customer Engagement — The Basket Size Problem

**Business question:** Which customer segments buy the most per order, and what drives it?

In B2B MRO, basket size is the primary revenue lever — converting a single-item buyer into a multi-item buyer is far more profitable than acquiring a new customer.


In [ ]:
# Basket size and order value by technician level
basket_by_level = (
    customer_orders
    .groupby('technician_level')
    .agg(
        avg_basket   = ('basket_size',   'mean'),
        avg_value    = ('order_value',   'mean'),
        total_orders = ('order_id',      'count'),
        return_rate  = ('is_returned',   'mean')
    )
    .round(2)
    .sort_values('avg_value', ascending=False)
)
print(basket_by_level)


In [ ]:
# Same breakdown by work situation — Facility Managers buy for whole sites
basket_by_work = (
    customer_orders
    .groupby('work_situation')
    .agg(avg_basket=('basket_size','mean'), avg_value=('order_value','mean'))
    .sort_values('avg_value', ascending=False)
    .round(2)
)
print(basket_by_work)


### Findings — Customer Engagement

The customer engagement analysis shows a fairly clear pattern across technician levels. More experienced technicians tend to place larger and more complete orders. Maintenance Engineers average around 5.4 items per order, compared with 1.9 items for Junior Technicians.

At first glance, this probably reflects purchasing familiarity as much as demand volume. More experienced buyers appear more comfortable ordering complete sets upfront, while junior buyers are more likely to place smaller or fragmented orders.

Return rates are also slightly higher among the junior segments. The difference is not extreme, but in a B2B environment returns still matter because they create extra operational work and interrupt the purchasing cycle.

One practical improvement could be a guided purchasing flow or recommended product bundles for less experienced buyers. Even a simple compatibility guide may help reduce incomplete orders and unnecessary returns.

## 4. End-to-End Delivery Performance

In [ ]:
# On-time rate and average delay by city
city_delivery = (
    customer_orders
    .groupby('city')
    .agg(
        ontime_rate  = ('on_time',     'mean'),
        avg_delay    = ('delay_days',  'mean'),
        total_orders = ('order_id',    'count')
    )
    .assign(ontime_pct=lambda df: (df['ontime_rate'] * 100).round(1))
    .sort_values('ontime_pct')
)
print(city_delivery)


In [ ]:
# Late order delay distribution — how bad are delays when they happen?
late_order_delays = customer_orders[customer_orders['on_time'] == 0]['delay_days']
print(f"Mean delay (late orders)   : {late_order_delays.mean():.1f} days")
print(f"Median delay (late orders) : {late_order_delays.median():.1f} days")
print(f"Max delay                  : {late_order_delays.max():.0f} days")
print(f"Orders delayed >7 days     : {(late_order_delays > 7).sum():,} ({(late_order_delays > 7).mean()*100:.1f}%)")


### Findings — Delivery Performance

The city-level analysis shows relatively small differences in on-time delivery performance across locations. Rates range from 53.7% in Cologne to 57.0% in Stuttgart and Berlin, so the gap between the lowest and highest-performing cities is only 3.3 percentage points. No city performs particularly well here.

The more important issue is the overall delivery performance itself. Every city in the dataset remains below a 60% on-time rate, which suggests the problem is affecting the network more broadly rather than a few isolated locations.

I originally expected stronger regional differences, but the data points more toward a wider reliability issue across carriers or operational processes. This looks less like a route problem and more like a network consistency problem.

From an operational perspective, the focus should probably be on improving overall delivery reliability rather than targeting individual cities separately. Areas such as carrier management, SLA reviews, and logistics process coordination are likely to have the biggest impact.

## 5. SQL Analysis — Procurement Business Queries

In [ ]:
# DuckDB runs SQL directly on DataFrames — same syntax as any company database
db = duckdb.connect()
db.register('orders_tbl',        customer_orders)
db.register('supplier_orders_tbl', supplier_orders)
db.register('suppliers_tbl',     supplier_master)


### SQL Query 1 — Which cities show the highest basket growth year-on-year?

In [ ]:
basket_growth_query = """
    SELECT
        city,
        AVG(CASE WHEN order_year = 2022 THEN basket_size END) AS avg_basket_2022,
        AVG(CASE WHEN order_year = 2023 THEN basket_size END) AS avg_basket_2023,
        ROUND(
            (AVG(CASE WHEN order_year = 2023 THEN basket_size END) -
             AVG(CASE WHEN order_year = 2022 THEN basket_size END)) /
             AVG(CASE WHEN order_year = 2022 THEN basket_size END) * 100, 1
        ) AS growth_pct
    FROM orders_tbl
    GROUP BY city
    HAVING avg_basket_2022 IS NOT NULL AND avg_basket_2023 IS NOT NULL
    ORDER BY growth_pct DESC
"""
basket_growth = db.execute(basket_growth_query).df()
print(basket_growth)


### SQL Query 2 — High-risk suppliers: large order value + low reliability

In [ ]:
high_risk_supplier_query = """
    SELECT
        so.supplier_id,
        s.supplier_country,
        ROUND(s.reliability_score * 100, 1)  AS reliability_pct,
        COUNT(so.sup_order_id)               AS order_count,
        ROUND(SUM(so.order_value) / 1000, 0) AS total_value_k,
        ROUND(AVG(so.actual_lead_days), 1)   AS avg_lead_days
    FROM supplier_orders_tbl so
    JOIN suppliers_tbl s ON so.supplier_id = s.supplier_id
    WHERE s.reliability_score < 0.80
    GROUP BY so.supplier_id, s.supplier_country, s.reliability_score
    HAVING total_value_k > 50
    ORDER BY total_value_k DESC
    LIMIT 10
"""
high_risk_suppliers = db.execute(high_risk_supplier_query).df()
print(high_risk_suppliers)


### SQL Query 3 — Customer churn risk (no order in 90+ days)

In [ ]:
churn_risk_query = """
    WITH last_order AS (
        SELECT
            customer_id,
            MAX(order_date) AS last_order_date,
            COUNT(order_id)              AS lifetime_orders,
            SUM(order_value)             AS lifetime_value
        FROM orders_tbl
        GROUP BY customer_id
    )
    SELECT
        customer_id,
        CAST(last_order_date AS VARCHAR) AS last_order_date,
        lifetime_orders,
        ROUND(lifetime_value, 0)         AS lifetime_value,
        DATE_DIFF('day', last_order_date, DATE '2024-01-01') AS days_since_order
    FROM last_order
    WHERE DATE_DIFF('day', last_order_date, DATE '2024-01-01') > 90
    ORDER BY lifetime_value DESC
    LIMIT 15
"""
churn_risk_customers = db.execute(churn_risk_query).df()
print(f"Customers at churn risk: {len(churn_risk_customers)}")
print(churn_risk_customers.head(5))


### SQL Query 4 — Segment performance matrix (technician level × city)

In [ ]:
segment_matrix_query = """
    SELECT
        technician_level,
        city,
        COUNT(order_id)                         AS orders,
        ROUND(AVG(basket_size), 1)              AS avg_basket,
        ROUND(AVG(order_value), 0)              AS avg_value,
        ROUND(AVG(on_time) * 100, 1)            AS ontime_pct,
        ROUND(AVG(is_returned) * 100, 1)        AS return_pct
    FROM orders_tbl
    GROUP BY technician_level, city
    ORDER BY avg_value DESC
    LIMIT 15
"""
segment_matrix = db.execute(segment_matrix_query).df()
print(segment_matrix)


## 6. RFM Customer Segmentation

RFM is the industry standard method for B2B customer analysis — it scores each customer on:
- **Recency** — how recently did they order?
- **Frequency** — how often do they order?
- **Monetary** — how much have they spent?

Each dimension is scored 1–5, combined into a total, then grouped into business segments.


In [ ]:
# Build RFM table — one row per customer
snapshot_date = customer_orders['order_date'].max() + pd.Timedelta(days=1)

rfm_scores = (
    customer_orders
    .groupby('customer_id')
    .agg(
        recency   = ('order_date', lambda x: (snapshot_date - x.max()).days),
        frequency = ('order_id',   'count'),
        monetary  = ('order_value','sum')
    )
    .reset_index()
)

# Score each dimension 1–5 (5 = best)
rfm_scores['R_score'] = pd.qcut(rfm_scores['recency'],   5, labels=[5,4,3,2,1]).astype(int)
rfm_scores['F_score'] = pd.qcut(rfm_scores['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm_scores['M_score'] = pd.qcut(rfm_scores['monetary'],  5, labels=[1,2,3,4,5]).astype(int)
rfm_scores['RFM_total'] = rfm_scores['R_score'] + rfm_scores['F_score'] + rfm_scores['M_score']

def assign_rfm_label(row):
    t = row['RFM_total']
    if t >= 13: return 'Champion'
    elif t >= 11: return 'Loyal'
    elif t >= 9:  return 'Potential'
    elif t >= 7:  return 'New Customer'
    elif t >= 5:  return 'At Risk'
    else:         return 'Lost'

rfm_scores['segment'] = rfm_scores.apply(assign_rfm_label, axis=1)
print(rfm_scores['segment'].value_counts())
print(f"\nAt-risk revenue: €{rfm_scores[rfm_scores['segment']=='At Risk']['monetary'].sum():,.0f}")


### Findings — RFM Segments

The RFM analysis shows a familiar B2B pattern: a relatively small group of customers generates most of the revenue. Champions and Loyal customers account for a disproportionate share of sales despite representing fewer total accounts.

The At Risk segment stands out from a commercial perspective. These customers purchased regularly in the past but have become less active over time. Because they were previously engaged buyers, they are still realistic recovery opportunities compared with fully lost customers.

One practical implication is that retention efforts should not be spread evenly across all customer groups. A targeted re-engagement campaign for At Risk customers — for example after 60 days without a purchase — would likely produce better results than broad campaigns sent to the full customer base.

One limitation here is that the analysis focuses on revenue and activity patterns only. Customer profitability and support costs were not included in the segmentation.

## 7. K-Means Clustering — Behavioural Segmentation

RFM uses predefined rules to label customers. K-Means lets the algorithm find natural groups from the data itself.
The two approaches are complementary. RFM tells us what the segments *should* look like, K-Means tells us what patterns *actually exist*.


In [ ]:
# Features for clustering: recency, frequency, monetary
cluster_input = rfm_scores[['recency', 'frequency', 'monetary']].copy()
feature_scaler = StandardScaler()
scaled_features = feature_scaler.fit_transform(cluster_input)

# Test k=2 to k=5 — pick by silhouette score
silhouette_results = {}
for k in range(2, 6):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    predicted_labels = km.fit_predict(scaled_features)
    silhouette_results[k] = silhouette_score(scaled_features, predicted_labels)

best_k = max(silhouette_results, key=silhouette_results.get)
print(f"Silhouette scores: {silhouette_results}")
print(f"Optimal k: {best_k} (score: {silhouette_results[best_k]:.3f})")


In [ ]:
# Fit final model with optimal k
km_model = KMeans(n_clusters=best_k, random_state=42, n_init=10)
rfm_scores['cluster'] = km_model.fit_predict(scaled_features)

# Profile each cluster — sort by avg revenue to label them meaningfully
cluster_profiles = (
    rfm_scores.groupby('cluster')
    .agg(avg_recency=('recency','mean'), avg_frequency=('frequency','mean'),
         avg_monetary=('monetary','mean'), customer_count=('customer_id','count'))
    .sort_values('avg_monetary', ascending=False)
    .round(1)
)
print(cluster_profiles)


### Findings — Clustering

The clustering results show a fairly typical customer distribution: a small High Value group generates a large share of the business, while other clusters contribute far less consistently.

The High Value customers are clearly the most commercially important group in the dataset. Given their purchasing activity and revenue contribution, they would probably benefit more from closer account management and direct relationship support than from generic marketing campaigns.

The Dormant cluster needs a different approach. These customers were active before but now show low engagement levels. A targeted re-engagement campaign or even a simple follow-up call could help determine whether they are still potential customers or have already shifted to competitors.

Not every dormant account should necessarily be reactivated, though. For lower-value customers, the recovery cost may outweigh the expected return. That is something the business would need to evaluate before scaling retention campaigns.

## 8. Findings — Strategic Priorities

One thing that becomes clear across the analysis is that several operational issues are connected rather than isolated. Delivery reliability, customer retention, and purchasing behaviour all appear to influence each other across the customer lifecycle.

The delivery findings are probably the most operationally urgent. On-time performance remains consistently low across the network, and the delay patterns suggest broader process or carrier reliability issues rather than a few isolated route problems. Improving delivery consistency would likely have an impact beyond logistics alone, particularly for customer satisfaction and repeat purchasing behaviour.

The customer segmentation results also point to a fairly concentrated revenue structure. A relatively small group of high-value customers contributes a large share of total revenue, while the At Risk and Dormant segments represent potential retention concerns. In practice, this means retention efforts would probably deliver more value if they focused on high-contribution accounts rather than broad campaigns across the entire customer base.

Another point worth noting is the difference in purchasing behaviour between customer groups. More experienced buyers tend to place larger and more complete orders, while less experienced customers show smaller basket sizes and slightly higher return activity. This may indicate an opportunity to improve ordering guidance or product recommendation processes, especially for newer buyers.

Overall, the analysis suggests that operational improvements and customer management strategies should be closely connected rather than treated as separate initiatives.

---

## 9. Limitations & Further Analysis

- Add carrier-level shipment data to identify whether specific providers are driving delays
- Compare customer revenue with margin data to identify the most profitable segments, not only the highest-spending ones
- Test whether warehouse workload or seasonal peaks are contributing to delivery delays
- Build a simple churn prediction workflow for At Risk customers
- Evaluate whether category-level delivery SLAs would improve performance for bulky product groups


---
*Marziyeh Eslamparasti | Business Analyst | Hamburg, Germany*  
*[LinkedIn](https://linkedin.com/in/marziyeh-eslamparasti) . [GitHub](https://github.com/marziyeh-ba)*